# DATATHON 2026 — THE GRIDBREAKER
**Hosted by:** VinTelligence - VinUniversity Data Science & AI Club

---

## THÔNG TIN NHÓM
- **Tên đội thi**: Last Dance
- **Mã đội thi**: kVmzJpHWUFT6mv82wG4U
- **Thành viên**: 

| STT | Họ và tên        |  Vai trò   | Email |
| --- | ---------------- | ---------- | ----- |
| 1   | Bùi Lê Khôi      | Đội trưởng |       |
| 2   | Nguyễn Hà Anh    | Thành viên |       |
| 3   | Bùi Công Mậu     | Thành viên |       |
| 4   | Thái Hữu Thọ     | Thành viên |       |

---

## GIỚI THIỆU FILE NOTEBOOK

- **Mục đích**: Tiền xử lý dữ liệu trong **Phần 2 — Trực quan hoá và Phân tích Dữ liệu**

- **Nội dung trong file**:

    - 

- **Dữ liệu sử dụng**:
    - Dữ liệu được cung cấp bởi ban tổ chức, gồm các file csv được chia theo lớp.
    
---

Bài làm được trình bày từ sau dòng này.

---
---

### Import các libs và dependencies

In [1]:
import re
import os
import math

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl as pxl
import seaborn as sns
import statsmodels.api as sm
import datetime as dt

from datetime import timedelta
from scipy import stats
from math import sqrt

from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, pairwise_distances, silhouette_score, precision_score, recall_score, f1_score
from sklearn.metrics import RocCurveDisplay, roc_auc_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

import warnings
warnings.filterwarnings("ignore")

# Ensure plots appear in the notebook
%matplotlib inline

# Print last updated timestamp
import time
print(f"Cập nhật lần cuối vào thời điểm {time.asctime()}")

Cập nhật lần cuối vào thời điểm Wed Apr 29 11:40:08 2026


---

In [2]:
# Đọc tất cả data từ data/raw, lưu vào các dataframe trong một dictionary

data_dir = '../data/raw'
data_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
data_frames = {}
for file in data_files:
    file_path = os.path.join(data_dir, file)
    df_name = os.path.splitext(file)[0]
    data_frames[df_name] = pd.read_csv(file_path)

# Kiểm tra dữ liệu đã được đọc thành công
for name, df in data_frames.items():
    print(f"DataFrame '{name}' has {df.shape[0]} rows and {df.shape[1]} columns.")

DataFrame 'customers' has 121930 rows and 7 columns.
DataFrame 'geography' has 39948 rows and 4 columns.
DataFrame 'inventory' has 60247 rows and 17 columns.
DataFrame 'orders' has 646945 rows and 8 columns.
DataFrame 'order_items' has 714669 rows and 7 columns.
DataFrame 'payments' has 646945 rows and 4 columns.
DataFrame 'products' has 2412 rows and 8 columns.
DataFrame 'promotions' has 50 rows and 10 columns.
DataFrame 'returns' has 39939 rows and 7 columns.
DataFrame 'reviews' has 113551 rows and 7 columns.
DataFrame 'sales' has 3833 rows and 3 columns.
DataFrame 'sample_submission' has 548 rows and 3 columns.
DataFrame 'shipments' has 566067 rows and 4 columns.
DataFrame 'web_traffic' has 3652 rows and 7 columns.


In [3]:
# # Kiểm tra data
# for name, df in data_frames.items():
#     print(f"\nDataFrame '{name}' head:")
#     print(df.head())

In [4]:
# Kiểm tra thông tin từng DataFrame
for name, df in data_frames.items():
    print(f"\nDataFrame '{name}' info:")
    print(df.info())


DataFrame 'customers' info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121930 entries, 0 to 121929
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   customer_id          121930 non-null  int64 
 1   zip                  121930 non-null  int64 
 2   city                 121930 non-null  object
 3   signup_date          121930 non-null  object
 4   gender               121930 non-null  object
 5   age_group            121930 non-null  object
 6   acquisition_channel  121930 non-null  object
dtypes: int64(2), object(5)
memory usage: 6.5+ MB
None

DataFrame 'geography' info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39948 entries, 0 to 39947
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   zip       39948 non-null  int64 
 1   city      39948 non-null  object
 2   region    39948 non-null  object
 3   district  39948 non-null

In [5]:
# Kiểm tra số lượng null
for name, df in data_frames.items():
    print(f"\nDataFrame '{name}' null values:")
    print(df.isnull().sum())


DataFrame 'customers' null values:
customer_id            0
zip                    0
city                   0
signup_date            0
gender                 0
age_group              0
acquisition_channel    0
dtype: int64

DataFrame 'geography' null values:
zip         0
city        0
region      0
district    0
dtype: int64

DataFrame 'inventory' null values:
snapshot_date        0
product_id           0
stock_on_hand        0
units_received       0
units_sold           0
stockout_days        0
days_of_supply       0
fill_rate            0
stockout_flag        0
overstock_flag       0
reorder_flag         0
sell_through_rate    0
product_name         0
category             0
segment              0
year                 0
month                0
dtype: int64

DataFrame 'orders' null values:
order_id          0
order_date        0
customer_id       0
zip               0
order_status      0
payment_method    0
device_type       0
order_source      0
dtype: int64

DataFrame 'order_items' 

bảng promotions và order_items có null, còn lại 0 null

---

Fill missing value

In [6]:
# Fill missing value trong promotions
# Theo đề, nếu trường applicable_category để trống (null) thì có nghĩa là áp dụng cho tất cả mọi ngành hàng.
# Do đó ta sẽ điền cho các ô này giá trị "All"

data_frames['promotions']['applicable_category'] = data_frames['promotions']['applicable_category'].fillna('All')

# Trong order_items, nếu promo_id, promo_id_2 để trống (null) thì là đơn hàng không được áp dụng khuyến mãi nào. 
# Do đó ta sẽ điền cho các ô này giá trị No_promo

data_frames['order_items']['promo_id'] = data_frames['order_items']['promo_id'].fillna('No promotion')
data_frames['order_items']['promo_id_2'] = data_frames['order_items']['promo_id_2'].fillna('No promotion')

---

Label Cleaning

In [7]:
# Label Cleaning các cột dưới đây:
# Bảng customers: acquisition_channel
# Bảng orders: order_status, payment_method, device_type, order_source
# Bảng payments: payment_method
# Bảng products: color
# Bảng promotions: promo_type, promo_channel
# Bảng returns: return_reason
# Bảng web_traffic: traffic_source

label_cleaning_columns = {
    'customers': ['acquisition_channel'],
    'orders': ['order_status', 'payment_method', 'device_type', 'order_source'],
    'payments': ['payment_method'],
    'products': ['color'],
    'promotions': ['promo_type', 'promo_channel'],
    'returns': ['return_reason'],
    'web_traffic': ['traffic_source']
}

for df_name, cols in label_cleaning_columns.items():
    for col in cols:
        data_frames[df_name][col] = data_frames[df_name][col].str.replace('_', ' ').str.capitalize()

# Bảng geography có cột district cần lọc dấu #, như "District #01" -> "District 01"
data_frames['geography']['district'] = data_frames['geography']['district'].str.replace('#', '')

In [8]:
# Lưu lại vào data/processed

processed_dir = '../data/processed'
if not os.path.exists(processed_dir):
    os.makedirs(processed_dir)
for name, df in data_frames.items():
    df.to_csv(os.path.join(processed_dir, f"{name}.csv"), index=False)